# **ST 554 Spring 2026 Homework Seven**
## *Created by Cody Ashby on March 30, 2026*

This assignment allows us the opportunity to do one of my favorites things...modelling data!
We'll be comparing quite a handful of models in both the (multiple) linear and logistic sense. Regularization methods will also be applied as necessary.
The dataset we'll be using is the `wine-quality` dataset from UCI's Machine Learning Repository.
Before diving into this data, though, we'll need to import and install a bunch of modules.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.random import default_rng
from math import sqrt
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable


Notice that we needed to install `scikit-learn` before actually importing it. The latter is done below.

In [2]:
import sklearn

We'll be doing a bunch of things from this module, so let's import some extra things that allow us to perform both linear and logistic regression, splitting our data into training and testing sets, cross-validation, model selection, interaction and polynomial terms, and metrics.

In [3]:
from sklearn import linear_model
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV, Lasso, LassoCV, Ridge, RidgeCV, ElasticNet, ElasticNetCV
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, log_loss, accuracy_score

### Reading and Combining the Datasets

Now, let's read in our datasets, which are separated by wine color. We'll also look at a quick description for each, beginning with the red wine.

In [4]:
winequality_red=pd.read_csv("winequality-red.csv",sep=";")
winequality_red.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


Observe that the white wine has more than triple the amount of observations. This may indicate how important the type of wine is when it comes to modelling our data.

In [5]:
winequality_white=pd.read_csv("winequality-white.csv",sep=";")
winequality_white.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000
mean,6.854788,0.278241,0.334192,6.391415,0.045772,35.308085,138.360657,0.994027,3.188267,0.489847,10.514267,5.877909
std,0.843868,0.100795,0.121020,5.072058,0.021848,17.007137,42.498065,0.002991,0.151001,0.114126,1.230621,0.885639
min,3.800000,0.080000,0.000000,0.600000,0.009000,2.000000,9.000000,0.987110,2.720000,0.220000,8.000000,3.000000
25%,6.300000,0.210000,0.270000,1.700000,0.036000,23.000000,108.000000,0.991723,3.090000,0.410000,9.500000,5.000000
50%,6.800000,0.260000,0.320000,5.200000,0.043000,34.000000,134.000000,0.993740,3.180000,0.470000,10.400000,6.000000
75%,7.300000,0.320000,0.390000,9.900000,0.050000,46.000000,167.000000,0.996100,3.280000,0.550000,11.400000,6.000000
max,14.200000,1.100000,1.660000,65.800000,0.346000,289.000000,440.000000,1.038980,3.820000,1.080000,14.200000,9.000000


We'll combine these by using the `concat` function; a binary variable named `wine_type` is also introduced where the red wine is denoted as `0` while the white wine is assigned to `1`. This may be reiterated throughout the document.

In [6]:
winequality_combined=pd.concat([winequality_red,winequality_white],keys=['0','1']).reset_index(level=0).rename(columns={'level_0':'wine_type'})
winequality_combined

,wine_type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
1,0,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5
2,0,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5
3,0,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6
4,0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4893,1,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,1,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,1,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,1,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


Note that `wine_type` was added into the first column, which is not quite what we want. As a consequence, we'll need to rearrange these columns so that `wine_type` is last. Although the information in this dataframe isn't new, we'll assign a slightly different name here after rearranging the columns.

In [7]:
winequality_all=winequality_combined.iloc[:,[1,2,3,4,5,6,7,8,9,10,11,12,0]]
winequality_all

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,1
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,1
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,1
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,1


### Splitting the Data into Training and Testing Sets

Now, we get to dive into the fun part (at least for me). We'll utilize the `train_test_split` function from `sklearn` to split our data into training and testing sets. 80% will be used to fit our models, while the remainder will be saved for testing. Stratified sampling will be used to ensure that there's equal representation for both types of wine.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    winequality_all[['fixed acidity','volatile acidity','citric acid','residual sugar','chlorides',
                     'free sulfur dioxide','total sulfur dioxide','density','pH','sulphates','quality','wine_type']],
    winequality_all['alcohol'],
    test_size=0.20,
    random_state=100,
    shuffle=True,
    stratify=winequality_all['wine_type'])

### Modeling the Data Using Regression
#### *MLR Models*

Four different multiple linear regression models will be fit:

* One will include just a handful of the predictors abve.
* A second will use all of the predictors.
* A third will incorporate an interaction term.
* A fourth one will use some polynomial terms (without interaction, perhaps?).

Cross-validation will be used to select the best model here. Instances will be made using the `LinearRegression()` object from `sklearn`.
The binary variable `wine_type` will be considered in all cases.

In [9]:
mlr_fit=linear_model.LinearRegression()
mlr_one=mlr_fit.fit(X_train[['fixed acidity','residual sugar','chlorides','total sulfur dioxide','density','wine_type']],y_train)
print(np.round(mlr_one.intercept_,4),np.round(mlr_one.coef_,4))

584.3505 [ 3.361000e-01  1.798000e-01 -2.128100e+00 -1.100000e-03 -5.788136e+02
 -1.677500e+00]


This first model can be interpreted as...

`alcohol` = 584.3505 + 0.3361*`fixed acidity` + 0.1798*`residual sugar` - 2.1281*`chlorides` - 1.1*`total sulfur dioxide` - 578.8136*`density` - 1.6775*`wine_type`,

where `wine_type` = 0 for red wine and 1 for white wine.

In [10]:
mlr_two=mlr_fit.fit(X_train[['fixed acidity','volatile acidity','citric acid','residual sugar','chlorides',
                             'free sulfur dioxide','total sulfur dioxide','density','pH','sulphates','quality','wine_type']],
                    y_train)
print(np.round(mlr_two.intercept_,4),np.round(mlr_two.coef_,4))

644.4877 [ 5.208000e-01  8.237000e-01  5.532000e-01  2.283000e-01 -6.242000e-01
 -3.400000e-03 -2.000000e-04 -6.515311e+02  2.623200e+00  1.045200e+00
  1.080000e-01 -1.101100e+00]


If we use all the predictors...

Below, we define some interaction and polynomial terms.

In [11]:
poly1=PolynomialFeatures(interaction_only=True,include_bias=False)
poly2=PolynomialFeatures(include_bias=False)
design_inter1=poly1.fit_transform(X_train[['fixed acidity','pH']])
design_poly1=poly2.fit_transform(X_train[['residual sugar','quality']])

We'll incorporate the first transformation regarding the interaction between fixed acidity and pH.

In [12]:
mlr_three=mlr_fit.fit(np.hstack([design_inter1,X_train[['free sulfur dioxide','density','sulphates','wine_type']]]),y_train)
print(np.round(mlr_three.intercept_,4),np.round(mlr_three.coef_,4))

339.9589 [ 4.751000e-01  1.667000e+00 -6.800000e-02 -1.100000e-03 -3.386033e+02
  5.948000e-01 -1.983000e-01]


Next up is the second transformation using the polynomial terms for density and the wine type.

In [13]:
mlr_four=mlr_fit.fit(np.hstack([design_poly1,X_train[['density','wine_type']]]),y_train)
print(np.round(mlr_four.intercept_,4),np.round(mlr_four.coef_,4))

463.4705 [ 1.351000e-01 -1.852000e-01  3.400000e-03 -1.270000e-02  4.320000e-02
 -4.550728e+02 -1.652600e+00]


Now, we'll use cross-validation to find the best MLR model. Five folds will be used throughout. We'll start with the first one.

In [14]:
mlr_CV_one=cross_validate(mlr_one,
                          X_train[['fixed acidity','residual sugar','chlorides','total sulfur dioxide','density','wine_type']],
                          y_train, cv=5)
mlr_CV_one['test_score']

array([0.53074412, 0.72928019, 0.74164878, 0.77055057, 0.75122231])

The array of CV "test scores" is given above. Below is the average across that array.

In [15]:
mlr_CV_one['test_score'].mean()

0.7046891956705823

Next is the cross-validation for the second MLR model with all the predictors included.

In [16]:
mlr_CV_two=cross_validate(mlr_two,
                          X_train[['fixed acidity','volatile acidity','citric acid','residual sugar','chlorides',
                             'free sulfur dioxide','total sulfur dioxide','density','pH','sulphates','quality','wine_type']],
                          y_train,cv=5)
mlr_CV_two['test_score']

array([0.65652271, 0.83034432, 0.8465524 , 0.85792983, 0.85715334])

As you can see, all of those went up significantly. The average CV score increased as a result.

In [17]:
mlr_CV_two['test_score'].mean()

0.8097005203901478

We see some improvement in the cross-validation process when an interaction term is incorporated.

In [18]:
mlr_CV_three=cross_validate(mlr_three,np.hstack([design_inter1,X_train[['free sulfur dioxide','density','sulphates','wine_type']]]),y_train,cv=5)
mlr_CV_three['test_score']

array([0.40699898, 0.56114696, 0.61870841, 0.64822251, 0.61596224])

The average CV score is now closer to the middle.

In [19]:
mlr_CV_three['test_score'].mean()

0.5702078207621232

If we use polynomial terms (namely quadratics), the CV scores did increase a tad despite the barely noticeable dip from the first fold.

In [20]:
mlr_CV_four=cross_validate(mlr_four,np.hstack([design_poly1,X_train[['density','wine_type']]]),y_train,cv=5)
mlr_CV_four['test_score']

array([0.40688301, 0.66601434, 0.65958012, 0.6943847 , 0.69669259])

Still, the average CV score increased.

In [21]:
mlr_CV_four['test_score'].mean()

0.6247109525293842

By cross-validation, the best MLR model involves the interaction between the fixed acidity and pH, while also considering the amounts of free sulfur dioxide, sulphates, density and type of wine.

#### *LASSO Model*

Now, we'll try fitting a LASSO model with a different handful of predictors compared to the first MLR model.

In [22]:
LASSO_mod=LassoCV(cv=5,random_state=100,alphas=np.linspace(0,2,100)) \
                  .fit(X_train[['citric acid','chlorides','free sulfur dioxide','quality','wine_type']].values,y_train.values)

/home/jupyter-cashby@ncsu.edu/.local/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:681: UserWarning: Coordinate descent without L1 regularization may lead to unexpected results and is discouraged. Set l1_ratio > 0 to add L1 regularization.
  model = cd_fast.enet_coordinate_descent_gram(
/home/jupyter-cashby@ncsu.edu/.local/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 2153.0901875399704, tolerance: 0.5985134593435434
  model = cd_fast.enet_coordinate_descent_gram(
/home/jupyter-cashby@ncsu.edu/.local/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:681: UserWarning: Coordinate descent without L1 regularization may lead to unexpected results and is discouraged. Set l1_ratio > 0 to add L1 regularization.
  model = cd_fast.enet_coordinate_descent_gram(
/home/jupyter-cashby@ncsu.edu/.local/lib/pytho

Finding the optimal tuning parameter for the LASSO model appears to be fruitless in this case. This most likely illustrates the lack of necessity for applying an L_1 penalty.

In [23]:
print(LASSO_mod.alpha_)

0.0


Since the optimal tuning parameter here is `0`, this will be no different than our traditional least squares estimates.

In [24]:
LASSO_optimal=linear_model.Lasso(LASSO_mod.alpha_)
LASSO_optimal.fit(X_train[['citric acid','chlorides','free sulfur dioxide','quality','wine_type']],y_train)
print(LASSO_optimal.intercept_,LASSO_optimal.coef_)

8.228251483997743 [ 0.05198677 -8.10384522 -0.01576909  0.56066052 -0.08317516]


/home/jupyter-cashby@ncsu.edu/.local/lib/python3.9/site-packages/sklearn/base.py:1389: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/home/jupyter-cashby@ncsu.edu/.local/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:695: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
/home/jupyter-cashby@ncsu.edu/.local/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.684e+03, tolerance: 7.462e-01 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.li

#### *Ridge Regresson Model*

Let's see if using Ridge regression would make a difference.

In [25]:
Ridge_mod=RidgeCV(cv=5,alphas=np.linspace(0,5,100)).fit(X_train[['volatile acidity','residual sugar','density','sulphates','wine_type']],y_train)

Luckily, there were no warnings here since we're dealing with a different type of penalty.

In [26]:
print(Ridge_mod.alpha_)

0.0


Again, this parameters will be essentially the same as those from using least squares.

In [27]:
Ridge_optimal=linear_model.Ridge(Ridge_mod.alpha_)
Ridge_optimal.fit(X_train[['volatile acidity','residual sugar','density','sulphates','wine_type']],y_train)
print(Ridge_optimal.intercept_,Ridge_optimal.coef_)

517.2646530611233 [-1.10760313e-02  1.47795540e-01 -5.09703197e+02  1.27446204e+00
 -1.65345188e+00]


#### *Elastic Net Model*

For this type of model, we'll see if applying both types of penalties will make a difference.

In [28]:
ElasticNet_mod=ElasticNetCV(cv=5,random_state=10,l1_ratio=[0.1,0.25,0.4,0.5,0.6,0.75,0.8,0.85,0.9,0.95,0.96,0.97,0.98,1],n_alphas=50)
ElasticNet_mod.fit(X_train[['fixed acidity','citric acid','chlorides','pH','wine_type']],y_train)

ElasticNetCV(cv=5,
             l1_ratio=[0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.8, 0.85, 0.9, 0.95,
                       0.96, 0.97, 0.98, 1],
             n_alphas=50, random_state=10)

Although the L_1 ratios were fixed ahead of time, it appeared to be worthwhile to apply both penalties.

In [30]:
print(ElasticNet_mod.alpha_,ElasticNet_mod.l1_ratio_)

0.00015388886612338018 0.85


Now, we'll see how both penalties affected the model.

In [31]:
ElasticNet_optimal=ElasticNet(alpha=ElasticNet_mod.alpha_,l1_ratio=ElasticNet_mod.l1_ratio_)
ElasticNet_optimal.fit(X_train[['fixed acidity','citric acid','chlorides','pH','wine_type']],y_train)
print(ElasticNet_optimal.intercept_,ElasticNet_optimal.coef_)

9.061977818619148 [ -0.0599837    0.76996344 -10.78476121   0.78814919  -0.40234605]


#### *Comparing Regression Models*
##### Using Root Mean Squared Error

At this point, we'll use the traditional Root Mean Squared Error metric to see which of those models are the best.

In [32]:
design_test=poly1.fit_transform(X_test[['fixed acidity','pH']])
np.sqrt(mean_squared_error(y_test,mlr_three.predict(np.hstack([design_test,X_test[['free sulfur dioxide','density','sulphates','wine_type']]]))))

220.37187124485422

That is abnormally large, even with an interaction term!

Let's see what happens with the RMSE from the LASSO model...

In [33]:
np.sqrt(mean_squared_error(y_test,LASSO_optimal.predict(X_test[['citric acid','chlorides','free sulfur dioxide','quality','wine_type']])))

0.9761840597800007

Okay, now we're talking! Here's the RMSE for the Ridge model, which is an improvement from LASSO.

In [34]:
np.sqrt(mean_squared_error(y_test,Ridge_optimal.predict(X_test[['volatile acidity','residual sugar','density','sulphates','wine_type']])))

0.6658279518507967

Here's how the RMSE is affected after applying both L1 and L2 penalties.

In [35]:
np.sqrt(mean_squared_error(y_test,ElasticNet_optimal.predict(X_test[['fixed acidity','citric acid','chlorides','pH','wine_type']])))

1.1142739089724132

So far, ridge regresstion seems to be the way to go based off of the RMSE metric.

##### Using Mean Absolute Error

Now, we'll follow the same procedure using a different metric, namely the mean absolute error.

In [36]:
mean_absolute_error(y_test,mlr_three.predict(np.hstack([design_test,X_test[['free sulfur dioxide','density','sulphates','wine_type']]])))

211.88280004263174

Okay, same story as with the first RMSE earlier. The rest should go more smoothly.

In [37]:
mean_absolute_error(y_test,LASSO_optimal.predict(X_test[['citric acid','chlorides','free sulfur dioxide','quality','wine_type']]))

0.7860971178167188

Slightly better compared to the second RMSE. Here's the MAE for the Ridge model.

In [38]:
mean_absolute_error(y_test,Ridge_optimal.predict(X_test[['volatile acidity','residual sugar','density','sulphates','wine_type']]))

0.5179651375067773

Lastly, here's the MAE for the Elastic Net model.

In [39]:
mean_absolute_error(y_test,ElasticNet_mod.predict(X_test[['fixed acidity','citric acid','chlorides','pH','wine_type']]))

0.9231115693044817

So far, it looks like Ridge regression may be the way to go and using MAE as a metric appears to work better than using the traditional RMSE.

### Modeling the Data Using Classification (Logistic Regression)
#### *Multiple Logistic Regression Models*

Now, we'll use logistic regression to predict the type of wine. Because of the different response variable, we'll need to do another training/testing split.

In [40]:
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    winequality_all[['fixed acidity','volatile acidity','citric acid','residual sugar','chlorides',
                     'free sulfur dioxide','total sulfur dioxide','density','pH','sulphates','alcohol','quality']],
    winequality_all['wine_type'],
    test_size=0.20,
    random_state=100,
    shuffle=True,
    stratify=winequality_all['wine_type'])

Let's define our instance of logistic regression first. Then we can start modeling.

In [41]:
log_reg=LogisticRegression(max_iter=5000)
log_reg_one=log_reg.fit(X_train_log[['volatile acidity','residual sugar','chlorides','total sulfur dioxide','density','pH','alcohol','sulphates','quality']],y_train_log)
print(log_reg_one.intercept_,log_reg_one.coef_)

[4.47582147] [[-8.4705695   0.10545384 -4.24363582  0.0567389  -0.60575072 -2.82270363
   0.81350129 -7.2259221  -0.03300131]]


Below are two different transformations that we'll use.

In [42]:
design_log_inter=poly1.fit_transform(X_train_log[['fixed acidity','chlorides']])
design_log_poly=poly2.fit_transform(X_train_log[['citric acid','free sulfur dioxide']])

Now, we can fit a logistic regression model with an interaction term.

In [43]:
log_reg_two=log_reg.fit(np.hstack([design_log_inter,X_train_log[['density','alcohol']]]),y_train_log)
print(log_reg_two.intercept_,log_reg_two.coef_)

[13.02960693] [[-0.29599886 -0.54542896 -9.06727848 -0.73050446 -0.47731548]]


Here's a logistic regression fit with quadratic terms.

In [44]:
log_reg_three=log_reg.fit(np.hstack([design_log_poly,X_train_log[['pH','quality']]]),y_train_log)
print(log_reg_three.intercept_,log_reg_three.coef_)

[13.97919655] [[ 1.23053608e+00  7.26620679e-02 -6.22746869e+00  1.90900775e-01
  -3.62815257e-04 -5.08722938e+00  1.98890484e-01]]


We could also do a more complex type of transformation, namely an interaction between three variables.

In [45]:
design_log_inter2=poly1.fit_transform(X_train_log[['citric acid','residual sugar','alcohol']])
log_reg_four=log_reg.fit(np.hstack([design_log_inter2,X_train_log[['volatile acidity','total sulfur dioxide','quality']]]),y_train_log)
print(log_reg_four.intercept_,log_reg_four.coef_)

[-6.88082994] [[-7.88986226e-02  2.75206298e-01  8.52215307e-01 -2.78067874e-01
  -2.21614233e-01 -6.37037511e-04 -1.08285893e+01  5.96694906e-02
  -3.85871765e-01]]


Cross-validation will now be used to determine which of the above models is the best.

In [46]:
log_reg_CV_one=cross_validate(log_reg_one,X_train_log[['volatile acidity','residual sugar','chlorides','total sulfur dioxide','density','pH','alcohol','sulphates','quality']],y_train_log,cv=5)
log_reg_CV_one['test_score'].mean()

0.9763319019767527

For cross-validation of logistic models, the higher average CV score leads to a better fitting model. An additive logistic model looks promising so far.

In [47]:
log_reg_CV_two=cross_validate(log_reg_two,np.hstack([design_log_inter,X_train_log[['density','alcohol']]]),y_train_log,cv=5)
log_reg_CV_two['test_score'].mean()

0.8883967572369883

A basic interaction term lowered the average CV score here, so polynomial terms will be considered instead.

In [48]:
log_reg_CV_three=cross_validate(log_reg_three,np.hstack([design_log_poly,X_train_log[['pH','quality']]]),y_train_log,cv=5)
log_reg_CV_three['test_score'].mean()

0.8599180054786407

We see a slight dip in the average CV score when polynomial terms are included.
Could a more complicated interaction term (namely a three-way one) do any better? Only one way to find out.

In [49]:
log_reg_CV_four=cross_validate(log_reg_four,np.hstack([design_log_inter2,X_train_log[['volatile acidity','total sulfur dioxide','quality']]]),y_train_log,cv=5)
log_reg_CV_four['test_score'].mean()

0.959976493669949

Close, but not quite. This means that out of the four multiple logistic regression models, the first model with no interaction or polynomial terms is slightly better than one with a three-way interaction term. Quite interesting!

#### *Logistic LASSO Model*

Here, we begin to see what applying penalties to the logistic regression model will do.

In [50]:
log_lasso_cv=LogisticRegressionCV(cv=5,solver='liblinear',penalty='l1',scoring='neg_log_loss',random_state=0)
log_lasso_train=log_lasso_cv.fit(X_train_log[['fixed acidity','chlorides','total sulfur dioxide','sulphates','quality']],y_train_log)
print(log_lasso_train.C_)

[21.5443469]


We'll use this as our regularization parameter to fit a logistic LASSO-type model.

In [51]:
log_lasso_optimal=LogisticRegression(solver='liblinear',penalty='l1',C=log_lasso_train.C_[0],random_state=0)
log_lasso_optimal_train=log_lasso_optimal.fit(X_train_log[['fixed acidity','chlorides','total sulfur dioxide','sulphates','quality']],y_train_log)
print(log_lasso_optimal_train.intercept_,log_lasso_optimal_train.coef_)

[4.97091931] [[ -0.66336531 -45.68406789   0.06175134  -9.45700914   0.61535644]]


#### *Logistic Ridge Model*

This can also be done utilizing an L_2 penalty.

In [52]:
log_ridge_cv=LogisticRegressionCV(cv=5,solver='newton-cg',penalty='l2',scoring='neg_log_loss',random_state=0)
log_ridge_train=log_ridge_cv.fit(X_train_log[['citric acid','free sulfur dioxide','density','pH','alcohol']],y_train_log)
print(log_ridge_train.C_)

[1291.54966501]


What a huge parameter! Regardless, we still need to use it to fit a model.

In [53]:
log_ridge_optimal=LogisticRegression(solver='newton-cg',penalty='l2',C=log_ridge_train.C_[0],random_state=0)
log_ridge_optimal_train=log_ridge_optimal.fit(X_train_log[['citric acid','free sulfur dioxide','density','pH','alcohol']],y_train_log)
print(log_ridge_optimal_train.intercept_,log_ridge_optimal_train.coef_)

[433.22986823] [[ 2.02516671e+00  1.14123930e-01 -4.18487675e+02 -5.06209709e+00
  -2.51087329e-01]]


#### *Logistic Elastic Net Model*

Here's what both penalties will do in a logistic regression model. As a consequence, a different type of numerical solver will need to be used, which may lead to a longer computational time.

In [63]:
log_ElasticNet_cv=LogisticRegressionCV(cv=5,solver='saga',penalty='elasticnet',
                                       l1_ratios=[0.05,0.1,0.2,0.25,0.3,0.4,0.5,0.6,0.7,0.75,0.8,0.9,0.95],
                                       max_iter=2500,scoring='neg_log_loss',random_state=0)
log_ElasticNet_train=log_ElasticNet_cv.fit(X_train_log[['volatile acidity','residual sugar','density','sulphates','quality']],y_train_log)
print(log_ElasticNet_train.l1_ratio_)

[0.95]


No matter how many iterations were using in fitting this model, the L_1 ratio still turned out to be 0.95. We'll now use it to fit our logistic Elastic Net model.

In [64]:
log_ElasticNet_optimal=LogisticRegression(solver='saga',penalty='elasticnet',l1_ratio=log_ElasticNet_train.l1_ratio_[0],random_state=0)
log_ElasticNet_optimal_train=log_ElasticNet_optimal.fit(X_train_log[['volatile acidity','residual sugar','density','sulphates','quality']],y_train_log)
print(log_ElasticNet_optimal_train.intercept_,log_ElasticNet_optimal_train.coef_)

[0.54701464] [[-5.07915929  0.34949405  0.42277127 -3.66958008  0.47345212]]


/home/jupyter-cashby@ncsu.edu/.local/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


#### *Comparing Classification Models*
##### Using Log-Loss

We'll now see which of these models is the best; first, the log-loss metric will be used to determine which type of wine is most likely to be chosen. The closer to `0`, the more likely a red wine was chosen.

In [65]:
log_reg_one_proba_preds=log_reg_one.predict_proba(X_test_log[['volatile acidity','residual sugar','chlorides','total sulfur dioxide','density','pH','alcohol','sulphates','quality']])
log_loss(y_test_log,log_reg_one_proba_preds)

/home/jupyter-cashby@ncsu.edu/.local/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


27.171369477949852

That made no sense whatsoever, in the context of probability.

Here's how log-loss would be affected if we apply one penalty at a time.

In [66]:
log_lasso_proba_preds=log_lasso_optimal.predict_proba(X_test_log[['fixed acidity','chlorides','total sulfur dioxide','sulphates','quality']])
log_loss(y_test_log,log_lasso_proba_preds)

0.10194227685073394

Definitely looks like a red wine. Here's the Ridge log-loss.

In [67]:
log_ridge_proba_preds=log_ridge_optimal.predict_proba(X_test_log[['citric acid','free sulfur dioxide','density','pH','alcohol']])
log_loss(y_test_log,log_ridge_proba_preds)

0.2360156101102889

That's quite an increase, even though it's still likely to be a red wine. Let's see what both penalties do here.

In [68]:
log_ElasticNet_proba_preds=log_ElasticNet_optimal.predict_proba(X_test_log[['volatile acidity','residual sugar','density','sulphates','quality']])
log_loss(y_test_log,log_ElasticNet_proba_preds)

0.25452125875814635

The Elastic Net has the larger (more sensible) log-loss, so that would be the best model to use here.

##### Using Accuracy Score

We can also measure how accurate the predictions are.

In [69]:
log_reg_one_preds=log_reg_one.predict(X_test_log[['volatile acidity','residual sugar','chlorides','total sulfur dioxide','density','pH','alcohol','sulphates','quality']])
accuracy_score(y_test_log,log_reg_one_preds)

/home/jupyter-cashby@ncsu.edu/.local/lib/python3.9/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


0.24615384615384617

25% accuracy is not good. Let's see what LASSO does.

In [70]:
log_lasso_preds=log_lasso_optimal.predict(X_test_log[['fixed acidity','chlorides','total sulfur dioxide','sulphates','quality']])
accuracy_score(y_test_log,log_lasso_preds)

0.9684615384615385

There we go! Let's see what happens with Ridge.

In [71]:
log_ridge_preds=log_ridge_optimal.predict(X_test_log[['citric acid','free sulfur dioxide','density','pH','alcohol']])
accuracy_score(y_test_log,log_ridge_preds)

0.9146153846153846

Here's what both penalties would do to accuracy.

In [72]:
log_ElasticNet_preds=log_ElasticNet_optimal.predict(X_test_log[['volatile acidity','residual sugar','density','sulphates','quality']])
accuracy_score(y_test_log,log_ElasticNet_preds)

0.9146153846153846

It's weird how both the logistic Ridge and Elastic Net models have the exact same accuracy score. Since it's still a little less than that from the logistic LASSO model, the best logistic model to use in order to get the most accurate predictions would be to use the logistic LASSO model.

# **ST 554 Spring 2026 Homework Eight**
## *Created by Cody Ashby on April 4, 2026*
### Modeling the Data Using Tree-Based Models

This portion will focus on using trees to do one of two things:

- determine the average value of alcohol content based on criterion placed on a handful of the remaining variables (via a regression tree), or
- determine the type of wine that's most likely to be selected based on similar criterion (via a classification tree).

As always, we'll import the modules we need from `scikit-learn` before going any further.

In [73]:
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import GridSearchCV

We'll first do our regreesion tree models to predict the average amount of alcohol content.
Before proceeding, let's split our data into training and testing sets.

In [74]:
X_train, X_test, y_train, y_test = train_test_split(
    winequality_all[['fixed acidity','volatile acidity','citric acid','residual sugar','chlorides',
                     'free sulfur dioxide','total sulfur dioxide','density','pH','sulphates','quality','wine_type']],
    winequality_all['alcohol'],
    test_size=0.20,
    random_state=120,
    shuffle=True,
    stratify=winequality_all['wine_type'])

Below is the creation of an instance for our regression tree as well as a collection of values to test our parameters with. We'll then fit the model to get the best parameters to use for our tree.

In [77]:
parms={'max_depth':range(5,15),
       'min_samples_leaf':[6,7,8,9,10,11,12]}
regtree1=GridSearchCV(DecisionTreeRegressor(),parms,cv=5)
regtree1.fit(X_train[['fixed acidity','residual sugar','density','quality','wine_type']],y_train)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(),
             param_grid={'max_depth': range(5, 15),
                         'min_samples_leaf': [6, 7, 8, 9, 10, 11, 12]})

The best default score is shown below, as well as the best paramters to use (i.e. number of splits and the number of samples per leaf).

In [78]:
print(sqrt(regtree1.best_score_),regtree1.best_params_)

0.8900201422061017 {'max_depth': 12, 'min_samples_leaf': 7}


This means that the best regression tree for this model will have 12 splits and each leaf will have a minimum sample size of seven.

In [80]:
regtree1_optimal=DecisionTreeRegressor(max_depth=12,min_samples_leaf=7) \
                             .fit(X_train[['fixed acidity','residual sugar','density','quality','wine_type']],y_train)
regtree1_optimal.feature_importances_

array([0.05529655, 0.14242973, 0.68060016, 0.0372964 , 0.08437716])

Here, the most prominent variable in this model is the density, with the residual sugar as a distant second.

A grid to find the best parameters to use for a random forest of regression trees is shown below, as well as the end result.

In [91]:
parms2={'max_features': range(1,6), 'max_depth': range(10,26), 'min_samples_leaf': range(5,11)}
regforest1=GridSearchCV(RandomForestRegressor(),parms2,cv=5)
regforest1.fit(X_train[['citric acid','chlorides','total sulfur dioxide','pH','wine_type']],y_train)

GridSearchCV(cv=5, estimator=RandomForestRegressor(),
             param_grid={'max_depth': range(10, 26),
                         'max_features': range(1, 6),
                         'min_samples_leaf': range(5, 11)})

As before, the best default score, as well as the best parameters are shown below.

In [94]:
print(sqrt(regforest1.best_score_),regforest1.best_params_)

0.7003337775271763 {'max_depth': 14, 'max_features': 3, 'min_samples_leaf': 5}


In [95]:
regforest1_optimal=RandomForestRegressor(max_depth=14,max_features=3,min_samples_leaf=5) \
                    .fit(X_train[['citric acid','chlorides','total sulfur dioxide','pH','wine_type']],y_train)
regforest1_optimal.feature_importances_

array([0.14481909, 0.4305055 , 0.25897361, 0.15339121, 0.01231059])

Even though the amount of `chlorides` appears to be the most significant variable in fitting this random forest, it never reached a majority. The amount of `total sulfur dioxide` came in at a not-so-distant second with the `pH` and amount of `citric acid` not that far behind.

With these two newer models, let's see how they perform on the usual RMSE and MAE metrics.

In [85]:
sqrt(mean_squared_error(y_test,regtree1_optimal.predict(X_test[['fixed acidity','residual sugar','density','quality','wine_type']])))

0.542419210168694

The regression tree has a lower RMSE compared to the previous linear regression models, which implies that a regression tree would be a better representation for this data.

In [86]:
mean_absolute_error(y_test,regtree1_optimal.predict(X_test[['fixed acidity','residual sugar','density','quality','wine_type']]))

0.3959003119458332

A similar thing happens with the MAE.

In [96]:
sqrt(mean_squared_error(y_test,regforest1_optimal.predict(X_test[['citric acid','chlorides','total sulfur dioxide','pH','wine_type']])))

0.8496680608906403

The RMSE for the random forest increased significantly compared to that of the regresstion tree. The MAE may also spike up along with it.

In [97]:
mean_absolute_error(y_test,regforest1_optimal.predict(X_test[['citric acid','chlorides','total sulfur dioxide','pH','wine_type']]))

0.6495143512034252

Just as anticipated, the best ***linear*** route would be to predict the average amount of alcohol based on conditions placed on the `fixed acidity`, `residual sugar`, `density`, the `pH`, and the type of wine chosen.

Now, we'll do some classification trees. Another training and testing split will be performed since we're focused on a binary outcome.

In [87]:
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    winequality_all[['fixed acidity','volatile acidity','citric acid','residual sugar','chlorides',
                     'free sulfur dioxide','total sulfur dioxide','density','pH','sulphates','quality','alcohol']],
    winequality_all['wine_type'],
    test_size=0.20,
    random_state=120,
    shuffle=True,
    stratify=winequality_all['wine_type'])

Below is a grid that consists of parameters to test in fitting the classification tree.

In [88]:
parms3={'max_depth':range(5,15),
       'min_samples_leaf':range(5,31)}
classtree1=GridSearchCV(DecisionTreeClassifier(criterion='log_loss'),parms,cv=5)
classtree1.fit(X_train_class[['volatile acidity','chlorides','pH','sulphates','alcohol']],y_train_class)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(criterion='log_loss'),
             param_grid={'max_depth': range(5, 15),
                         'min_samples_leaf': [6, 7, 8, 9, 10, 11, 12]})

The best parameters to use, along with the best log-loss score, are shown below.

In [89]:
print(classtree1.best_score_,classtree1.best_params_)

0.9665181017250315 {'max_depth': 7, 'min_samples_leaf': 7}


The above parameters will now be used to fit our classification tree.

In [90]:
classtree1_optimal=DecisionTreeClassifier(max_depth=7,min_samples_leaf=7,criterion='log_loss') \
                    .fit(X_train_class[['volatile acidity','chlorides','pH','sulphates','alcohol']],y_train_class)
classtree1_optimal.feature_importances_

array([0.12093274, 0.66551202, 0.05776556, 0.12909608, 0.0266936 ])

The runaway favorite for most significant predictor in this tree is the amount of `chlorides`. Both `sulphates` and `volatile acidity`, although distant, are the next two significant.

Lastly, we'll fit a random forest based on several classification trees. A grid for testing the parameters and fitting a model is shown below.

In [99]:
parms4={'max_features':range(2,7),
        'max_depth':range(5,15),
       'min_samples_leaf':range(5,31)}
classforest1=GridSearchCV(RandomForestClassifier(criterion='log_loss'),parms,cv=5)
classforest1.fit(X_train_class[['fixed acidity','residual sugar','free sulfur dioxide','density','quality']],y_train_class)

GridSearchCV(cv=5, estimator=RandomForestClassifier(criterion='log_loss'),
             param_grid={'max_depth': range(5, 15),
                         'min_samples_leaf': [6, 7, 8, 9, 10, 11, 12]})

Here are the collection of parameters associated with the model that has the best log-loss value.

In [100]:
print(classforest1.best_score_,classforest1.best_params_)

0.9657512771155699 {'max_depth': 12, 'min_samples_leaf': 6}


Strange to see nothing chosen for the number of features to use. We'll use the rule of thumb where we take the square root of the number of predictors in the model. Rounded to the nearest integer, we'll use two features.

In [106]:
classforest1_optimal=RandomForestClassifier(max_features=2,max_depth=10,min_samples_leaf=6,criterion='log_loss') \
                    .fit(X_train_class[['fixed acidity','residual sugar','free sulfur dioxide','density','quality']],y_train_class)
classforest1_optimal.feature_importances_

array([0.10593456, 0.31763203, 0.19863305, 0.36021928, 0.01758107])

No predictor got a majority, but `density` has the largest plurality of importance with `residual sugar` not too far behind.

Let's see which one of these newer classification models performs the best. Both log-loss and accuracy scores will be considered.

In [92]:
classtree1_proba_preds=classtree1_optimal.predict_proba(X_test_class[['volatile acidity','chlorides','pH','sulphates','alcohol']])
log_loss(y_test_class,classtree1_proba_preds)

0.3918849582694388

This indicates a moderate chance that a red wine was chosen.

In [46]:
classtree1_preds=classtree1_optimal.predict(X_test_class[['volatile acidity','chlorides','pH','sulphates','alcohol']])
accuracy_score(y_test_class,classtree1_preds)

0.9638461538461538

The accuracy in predicting the type of wine is about a half of a percent lower than that from the logistic LASSO model done earlier.

In [107]:
classforest1_proba_preds=classforest1_optimal.predict_proba(X_test_class[['fixed acidity','residual sugar','free sulfur dioxide','density','quality']])
log_loss(y_test_class,classforest1_proba_preds)

0.1057881557292711

The `log-loss` criterion for the classification forest is smaller, indicating a stronger chance of randomly choosing a red wine.

In [108]:
classforest1_preds=classforest1_optimal.predict(X_test_class[['fixed acidity','residual sugar','free sulfur dioxide','density','quality']])
accuracy_score(y_test_class,classforest1_preds)

0.9623076923076923

Both of the newer classification models are quite strong in making accurate predictions. However, the logistic LASSO would be the best ***classification*** model to use due to the slightly lower lo